In [ ]:
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from pathlib import Path
from torchvision import transforms as T

import torchvision
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor



In [11]:
class PolesDataset(Dataset):
    def __init__(self, txt_file, dataset_root, transforms=None):
        self.root = Path(dataset_root)
        self.transforms = transforms
        with open(self.root / txt_file) as f:
            self.img_paths = [l.strip() for l in f if l.strip()]

    def __len__(self):
        return len(self.img_paths)

    def __getitem__(self, idx):
        # Load image
        img_path = self.root / self.img_paths[idx]
        img = Image.open(img_path).convert("RGB")
        W, H = img.size

        # Load label — YOLO format: class cx cy w h (normalised)
        lbl_path = img_path.with_suffix(".txt").as_posix()
        lbl_path = Path(lbl_path.replace("images", "labels"))

        boxes  = []
        labels = []
        if lbl_path.exists():
            with open(lbl_path) as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) == 5:
                        _, cx, cy, bw, bh = map(float, parts)
                        # Convert to absolute x1 y1 x2 y2
                        x1 = (cx - bw / 2) * W
                        y1 = (cy - bh / 2) * H
                        x2 = (cx + bw / 2) * W
                        y2 = (cy + bh / 2) * H
                        boxes.append([x1, y1, x2, y2])
                        labels.append(1)  # 1 = pole (0 is background in RCNN)

        # Handle empty images
        if len(boxes) == 0:
            boxes  = torch.zeros((0, 4), dtype=torch.float32)
            labels = torch.zeros((0,),   dtype=torch.int64)
        else:
            boxes  = torch.tensor(boxes,  dtype=torch.float32)
            labels = torch.tensor(labels, dtype=torch.int64)

        target = {"boxes": boxes, "labels": labels}

        if self.transforms:
            img = self.transforms(img)

        return img, target

In [12]:
transform = T.Compose([T.ToTensor()])

DATASET_ROOT = "Poles2025/Road_poles_iPhone"

train_dataset = PolesDataset("Train.txt",      DATASET_ROOT, transforms=transform)
val_dataset   = PolesDataset("Validation.txt", DATASET_ROOT, transforms=transform)

# collate_fn needed because images have different numbers of boxes
def collate_fn(batch):
    return tuple(zip(*batch))

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True,  collate_fn=collate_fn)
val_loader   = DataLoader(val_dataset,   batch_size=4, shuffle=False, collate_fn=collate_fn)

print(f"Train: {len(train_dataset)} images")
print(f"Val:   {len(val_dataset)} images")

Train: 942 images
Val:   261 images
